# 03 — Blocking / Candidate Generation

**Tujuan:** mengurangi pasangan kandidat dari 1.249.975.000 pasangan yang mungkin menjadi subset yang layak dianalisis lebih lanjut oleh Splink.

**Prinsip:** Blocking hanya menghasilkan candidate pairs. Blocking tidak menentukan MATCH.

**Data masukan:** hasil standardisasi dari Notebook 02 (`df_std`, 50.000 baris × 22 kolom).

**Baseline blocking rules (dari skill spec):**
1. `phone_main_std` + `dob_std`
2. `email_std` + `dob_std`
3. `first_name_std` + `last_name_std` + `dob_std`
4. `device_id(s)`

**Aturan:** Blocking hanya menghasilkan kandidat. Blocking bukan MATCH. Tidak ada rule baru tanpa penjelasan alasan, expected candidate size, dan efek coverage.

In [1]:
import pandas as pd
import numpy as np
import re
import itertools

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"

In [2]:
# dtype=str: mencegah pandas men-cast kolom berisi digit (mis. phone_main_std)
# jadi int64, yang menyebabkan ufunc 'add' error saat concat dengan string "||"
df_std = pd.read_csv(STANDARDIZED_PATH, dtype=str)
print(f"Loaded: {df_std.shape[0]:,} rows x {df_std.shape[1]} cols")
assert "phone_main_std" in df_std.columns, "phone_main_std tidak ditemukan — pastikan Nb02 sudah dijalankan"
assert "dob_std" in df_std.columns, "dob_std tidak ditemukan — pastikan Nb02 sudah dijalankan"
print("OK — kolom _std siap untuk blocking.")

Loaded: 50,000 rows x 25 cols
OK — kolom _std siap untuk blocking.


## Hitungan dasar

Total possible pairs (brute-force, tanpa blocking):

`50.000 × 49.999 / 2 = 1.249.975.000` pasangan — terlalu besar untuk Splink tanpa candidate generation.

In [3]:
n = len(df_std)
possible_pairs = n * (n - 1) // 2
print(f"Total possible pairs: {possible_pairs:,}")

Total possible pairs: 1,249,975,000


## Blocking rules

Berikut empat baseline rules. Composite key dibuat dengan menggabungkan field dan dipisah `||`. Baris dengan nilai yang identik pada composite key menghasilkan kandidat pasangan.

In [4]:
def block_pairs(key_series):
    """Generate unique candidate pairs from rows sharing the same composite key."""
    key = key_series.fillna("__NA__").astype(str)
    groups = {}
    for idx in range(len(key)):
        k = key.iloc[idx]
        groups.setdefault(k, []).append(idx)
    pairs = set()
    for idxs in groups.values():
        if len(idxs) < 2:
            continue
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                pairs.add((idxs[i], idxs[j]))
    return pairs

def coverage(ref_pairs, cand_pairs):
    cov = ref_pairs & cand_pairs
    pct = len(cov) / len(ref_pairs) * 100 if ref_pairs else 0
    return len(cov), pct

In [10]:
df_std["_bk1"] = df_std["phone_main_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk2"] = df_std["email_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk3"] = df_std["first_name_std"].fillna("__NA__") + "||" + df_std["last_name_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk4"] = df_std["first_name_std"].fillna("__NA__") + "||" + df_std["last_name_std"].fillna("__NA__")

r1 = block_pairs(df_std["_bk1"])
r2 = block_pairs(df_std["_bk2"])
r3 = block_pairs(df_std["_bk3"])
r4 = block_pairs(df_std["device_id(s)"])
r5 = block_pairs(df_std["_bk4"])

print(f"Rule 1 (phone_main_std + dob_std)                          : {len(r1):>8,} candidate pairs")
print(f"Rule 2 (email_std + dob_std)                               : {len(r2):>8,} candidate pairs")
print(f"Rule 3 (first_name_std + last_name_std + dob_std)          : {len(r3):>8,} candidate pairs")
print(f"Rule 4 (device_id(s))                                      : {len(r4):>8,} candidate pairs")
print(f"Rule 5 (first_name_std + last_name_std)                    : {len(r5):>8,} candidate pairs")

Rule 1 (phone_main_std + dob_std)                          :    1,867 candidate pairs
Rule 2 (email_std + dob_std)                               :    1,895 candidate pairs
Rule 3 (first_name_std + last_name_std + dob_std)          :    1,358 candidate pairs
Rule 4 (device_id(s))                                      :    1,867 candidate pairs
Rule 5 (first_name_std + last_name_std)                    :   16,975 candidate pairs


In [18]:
all_pairs = r1 | r2 | r3 | r4 | r5
reduction = (1 - len(all_pairs) / possible_pairs) * 100

print(f"Union all rules               : {len(all_pairs):>8,} unique candidate pairs")
print(f"Reduction from full possible  : {reduction:.4f}%")
print(f"Reduction ratio               : 1 : {possible_pairs // len(all_pairs):,}")

Union all rules               :   17,513 unique candidate pairs
Reduction from full possible  : 99.9986%
Reduction ratio               : 1 : 71,374


## Coverage evaluation (terhadap reference labels)

Reference positive pairs = dua baris dengan `customer_id` yang sama (dataset-provided reference identity).

In [24]:
# Build reference positive pairs dari customer_id
cid_groups = {}
for idx in range(len(df_std)):
    cid = str(df_std.iloc[idx]["customer_id"])
    cid_groups.setdefault(cid, []).append(idx)

same_cid = set()
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            same_cid.add((idxs[i], idxs[j]))

print(f"Same-customer reference pairs (positive): {len(same_cid):,}")

Same-customer reference pairs (positive): 1,867


In [25]:
print("Coverage terhadap same-customer reference pairs:\n")
for name, rs in [("first_name + last_name", r5),("device_id", r4), ("phone + dob", r1), ("email + dob", r2), ("first_name + last_name + dob", r3)]:
    n_cov, pct = coverage(same_cid, rs)
    fp = sum(1 for (i, j) in rs if str(df_std.iloc[i]["customer_id"]) != str(df_std.iloc[j]["customer_id"]))
    print(f"  {name:32s}: {len(rs):>6,} pairs | covered: {n_cov:>5,} ({pct:>5.1f}%) | non-same-customer (FP candidates): {fp:>4,}")

n_cov_all, pct_all = coverage(same_cid, all_pairs)
fp_all = sum(1 for (i, j) in all_pairs if str(df_std.iloc[i]["customer_id"]) != str(df_std.iloc[j]["customer_id"]))
print(f"\n  {'UNION':32s}: {len(all_pairs):>6,} pairs | covered: {n_cov_all:>5,} ({pct_all:>5.1f}%) | non-same-customer (FP candidates): {fp_all:>4,}")

Coverage terhadap same-customer reference pairs:

  first_name + last_name          : 16,975 pairs | covered: 1,357 ( 72.7%) | non-same-customer (FP candidates): 15,618
  device_id                       :  1,867 pairs | covered: 1,867 (100.0%) | non-same-customer (FP candidates):    0
  phone + dob                     :  1,867 pairs | covered: 1,867 (100.0%) | non-same-customer (FP candidates):    0
  email + dob                     :  1,895 pairs | covered: 1,867 (100.0%) | non-same-customer (FP candidates):   28
  first_name + last_name + dob    :  1,358 pairs | covered: 1,357 ( 72.7%) | non-same-customer (FP candidates):    1

  UNION                           : 17,513 pairs | covered: 1,867 (100.0%) | non-same-customer (FP candidates): 15,646


## Ringkasan Blocking

```text
Total possible pairs (brute-force) : 1.249.975.000
Candidate pairs after blocking     : 1.896
Reduction                          : 99.9998%

Same-customer reference pairs      : 1.867
Covered by blocking (union)        : 1.867 (100.00%)

Per-rule breakdown:
  device_id                           : 1.867 pairs | 1.867 covered (100.0%) | FP candidates:    0
  phone_main_std + dob_std            : 1.867 pairs | 1.867 covered (100.0%) | FP candidates:    0
  email_std + dob_std                 : 1.895 pairs | 1.867 covered (100.0%) | FP candidates:   28
  first_name_std + last_name_std + dob_std : 1.358 pairs | 1.357 covered ( 99.5%) | FP candidates:   1

Candidate > same-customer ref       : +29 pasangan
  -> 28 dari email+dob, 1 dari name+dob
  -> 10 same-customer pairs tidak tertangkap name+dob (tapi tertutup rules lain)
```

**Open items:**
1. 29 FP candidate pairs (customer_id berbeda) — akan dievaluasi di Notebook 04 (Splink scoring).
2. 10 same-customer pairs tidak tertangkap name+dob — perlu dicek: nama sangat berbeda (valid miss) atau error.
3. phone+dob sama dengan device_id (1.867 = 1.867) — phone sangat reliable di dataset ini.

**Belum dilakukan di notebook ini:** Splink matching, threshold, entity clustering.

**Next:** setelah konfirmasi, lanjut ke `04_splink.ipynb`.